# VRT Data Preparation

> Building a recall dataset from human-scored film memory reports.

In the Vigilance Reaction Task (VRT), participants watch a 12-minute film composed of 11 short clips — either emotionally distressing or neutral — and later perform a computerised task where film-related images occasionally appear on screen. When a clip comes to mind, the participant presses the spacebar and types what they remember.

The raw data has two layers. The **Excel exports** record every event in the task row by row: digits shown, background images, button presses, and typed text. The **scored documents** are Word files where human raters read each typed response and labelled which of the 11 clips it refers to, using tags like `[clip_three]` or `[invalid_press]`.

This notebook combines both layers into a single HDF5 dataset suitable for recall analyses (serial position curves, conditional response probability, etc.). We proceed in stages:

1. Parse the scored documents to get human-coded clip assignments
2. Parse the raw Excel files for event-stream data (cue images, timing)
3. Link the two via subject identifiers
4. Assemble the integer arrays and export

This notebook covers **stage 1** only.

## Setup

In [1]:
import re
import zipfile
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path

from jaxcmr.helpers import find_project_root

In [2]:
ROOT = Path(find_project_root())
SCORED_DIR = ROOT / "data" / "raw" / "Scored_VRT data files"
assert SCORED_DIR.is_dir(), f"Not found: {SCORED_DIR}"

## Reading a scored document

The scored files are `.docx` Word documents produced by [Taguette](https://www.taguette.org), a qualitative coding tool. Each file covers one participant's first VRT session. The filename encodes the rater and a 5-letter participant code, e.g. `Tommy_AGMWB_1st.docx`. Tommy scored all 240 participants; we use his files exclusively.

Inside, the text is a flat sequence of tagged recall reports. A `[button_press]` marker starts each report, followed by a clip label and the participant's typed text with interspersed detail tags:

```
[button_press] [clip_three] I remember kids playing football [Internal_tag]
in a garden [Internal_tag] and an accident happened [Internal_tag]
```

A `.docx` file is just a zip archive containing XML. We extract the paragraph text directly — no need for the `python-docx` library.

In [3]:
def read_docx_paragraphs(path: Path) -> list[str]:
    """Return the text of each paragraph in a .docx file."""
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    with zipfile.ZipFile(path) as z:
        tree = ET.parse(z.open("word/document.xml"))
    paragraphs = []
    for p in tree.findall(".//w:p", ns):
        texts = [t.text or "" for t in p.findall(".//w:t", ns)]
        paragraphs.append("".join(texts))
    return paragraphs

Let's read one file and look at what we get:

In [4]:
sample_path = SCORED_DIR / "Tommy_AGMWB_1st.docx"
sample_paragraphs = read_docx_paragraphs(sample_path)

for i, p in enumerate(sample_paragraphs):
    if p.strip():
        print(f"[{i}] {p}")

[0] Tommy_AGMWB_1st
[1] [button_press] [clip_nine] I remember someone performing a surgery on the skin of a person [Internal_tag]
[3] [button_press] [clip_six] I rememeber the genocide in Rwanda [Internal_tag] where 800.000 people were killed [Internal_tag]
[5] [button_press] [clip_three] I remember kids playing football [Internal_tag] in a garden [Internal_tag] and an accident happened with cars [Internal_tag] and one car killed one of the kids [Internal_tag]
[7] [button_press] [clip_eleven] An elephant attacking an injuring a person [Internal_tag] and the going outside the circus [Internal_tag] and attacking more people [Internal_tag]
[9] [button_press] [clip_eight] a couple in love sitting on a wall [Internal_tag] and a car crash happened [Internal_tag] and one of the cars killed the guy [Internal_tag]
[11] [button_press] [clip_four] someone with atool was trying to clean an eye [Internal_tag]
[13] [button_press] [clip_five] a couple was inside the car [Internal_tag] very happy [Int

Each non-empty paragraph is one button-press event. The first paragraph is just the filename header. Everything else follows the pattern:

```
[button_press] [clip_LABEL] <utterance text with [tag] markers>
```

## Parsing button presses

From each button-press block we extract three things:

1. **Clip labels** — which of the 11 clips the rater assigned (`clip_one` .. `clip_eleven`, `clip_unknown`, or `invalid_press`). These become the `recalls` array in the final HDF5.
2. **Order** — the sequence in which clips were recalled, which is what CRP and serial-position analyses operate on.
3. **Utterance text** — what the participant typed. We'll need this later to match 5-letter subject codes back to the numeric IDs in the raw Excel files, since the text appears in both sources.

Occasional multi-clip assignments (e.g., `[clip_one, clip_two]`) mean we can't just grab the first clip label per block. Instead we:

1. Join all paragraphs into one string (tags sometimes spill across paragraph breaks)
2. Split on `[button_press` to get one chunk per press
3. Within each chunk, find all clip labels and strip tag markers

In [5]:
CLIP_WORD_TO_NUMBER = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
    "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11,
}


@dataclass
class ButtonPress:
    """One scored button-press event from a VRT session."""
    clip_numbers: list[int]   # 1-11 for identified clips; empty for invalid
    clip_unknown: bool        # True if rater could not identify the clip
    invalid: bool             # True if rater marked as invalid_press
    text: str                 # Participant's typed response, tags stripped


# Patterns for clip labels and detail tags.
_CLIP_RE = re.compile(r"clip_(\w+)")
_TAG_RE = re.compile(r"\[(Internal_tag|External_tag|Repetition_tag|Error_tag|Other_tag)\]")
_BRACKET_RE = re.compile(r"\[[^\[\]]*\]")


def parse_button_presses(paragraphs: list[str]) -> list[ButtonPress]:
    """Parse all button-press events from a scored document's paragraphs."""
    # Join paragraphs with spaces — tags sometimes wrap across lines.
    full_text = " ".join(p.strip() for p in paragraphs)

    # Split on [button_press to isolate each event.
    # The first chunk is the header (filename), which we discard.
    chunks = re.split(r"\[button_press", full_text)
    if len(chunks) < 2:
        return []

    presses = []
    for chunk in chunks[1:]:
        invalid = "invalid_press" in chunk

        # Find all clip_X labels in this chunk.
        clip_numbers = []
        clip_unknown = False
        for match in _CLIP_RE.finditer(chunk):
            word = match.group(1)
            if word == "unknown":
                clip_unknown = True
            elif word in CLIP_WORD_TO_NUMBER:
                clip_numbers.append(CLIP_WORD_TO_NUMBER[word])

        # Strip all bracket markers to recover the plain utterance text.
        text = _BRACKET_RE.sub("", chunk)
        # Also strip the leading "] " or "] " left from the button_press bracket.
        text = re.sub(r"^[\]\s]+", "", text).strip()
        # Collapse whitespace.
        text = re.sub(r"\s+", " ", text)

        presses.append(ButtonPress(
            clip_numbers=clip_numbers,
            clip_unknown=clip_unknown,
            invalid=invalid,
            text=text,
        ))

    return presses

Let's try it on our sample file:

In [6]:
sample_presses = parse_button_presses(sample_paragraphs)

for i, bp in enumerate(sample_presses, 1):
    label = ", ".join(str(c) for c in bp.clip_numbers) or ("unknown" if bp.clip_unknown else "invalid")
    print(f"  {i:2d}. clip {label:>7s}  {bp.text[:80]}")

print(f"\n{len(sample_presses)} button presses, "
      f"{sum(1 for bp in sample_presses if bp.clip_numbers)} with clip labels")

   1. clip       9  I remember someone performing a surgery on the skin of a person
   2. clip       6  I rememeber the genocide in Rwanda where 800.000 people were killed
   3. clip       3  I remember kids playing football in a garden and an accident happened with cars 
   4. clip      11  An elephant attacking an injuring a person and the going outside the circus and 
   5. clip       8  a couple in love sitting on a wall and a car crash happened and one of the cars 
   6. clip       4  someone with atool was trying to clean an eye
   7. clip       5  a couple was inside the car very happy sitting on the back sit and then their ca
   8. clip       7  someone was in the sea sea trying to survive but he couldnt and I think he died
   9. clip       2  someone was shaving and while he was shaving he was bleeding

9 button presses, 9 with clip labels


That file had no multi-clip presses. Here's one that does — participant BQFXW has a button press tagged with two clips at once:

```
[button_press [clip_three, clip_eight]] drunk driving advert [Internal_tag]
moments before man gets crushed by car [Internal_tag, Error_tag]
```

The participant recalled content spanning clips 3 and 8 in a single response. Because `_CLIP_RE` finds *all* `clip_X` matches in the chunk, both end up in `clip_numbers`:

In [7]:
multi_paragraphs = read_docx_paragraphs(SCORED_DIR / "Tommy_BQFXW_1st.docx")
multi_presses = parse_button_presses(multi_paragraphs)

for i, bp in enumerate(multi_presses, 1):
    if len(bp.clip_numbers) > 1:
        print(f"Press {i}: clip_numbers={bp.clip_numbers}")
        print(f"  text: {bp.text}")
        break

Press 10: clip_numbers=[3, 8]
  text: drunk driving advert moments before man gets crushed by car


## Processing all scored files

Tommy scored all 240 participants. We glob his files and parse each one.

In [8]:
def subject_code_from_path(path: Path) -> str:
    """Return the 5-letter subject code from a scored .docx filename."""
    # e.g. Tommy_AGMWB_1st.docx -> "AGMWB"
    return path.stem.split("_")[1]

In [9]:
all_files = sorted(SCORED_DIR.glob("Tommy_*_1st.docx"))
print(f"{len(all_files)} scored files")

240 scored files


In [10]:
@dataclass
class ScoredSession:
    """Parsed data from one scored VRT document."""
    subject_code: str         # 5-letter participant code
    presses: list[ButtonPress]

    @property
    def recall_clips(self) -> list[int]:
        """Clip numbers in button-press order, including repeats."""
        clips = []
        for bp in self.presses:
            clips.extend(bp.clip_numbers)
        return clips

    @property
    def unique_clips(self) -> list[int]:
        """Clip numbers in first-mention order, no repeats."""
        seen = set()
        clips = []
        for c in self.recall_clips:
            if c not in seen:
                seen.add(c)
                clips.append(c)
        return clips

In [11]:
scored_sessions: dict[str, ScoredSession] = {}

for path in all_files:
    code = subject_code_from_path(path)
    paragraphs = read_docx_paragraphs(path)
    scored_sessions[code] = ScoredSession(
        subject_code=code, presses=parse_button_presses(paragraphs),
    )

print(f"{len(scored_sessions)} sessions")

240 sessions


### Summary statistics

In [12]:
import numpy as np

n_presses = [len(s.presses) for s in scored_sessions.values()]
n_valid = [sum(1 for bp in s.presses if bp.clip_numbers) for s in scored_sessions.values()]
n_invalid = [sum(1 for bp in s.presses if bp.invalid) for s in scored_sessions.values()]
n_unknown = [sum(1 for bp in s.presses if bp.clip_unknown) for s in scored_sessions.values()]
n_unique = [len(s.unique_clips) for s in scored_sessions.values()]
n_raw = [len(s.recall_clips) for s in scored_sessions.values()]

def describe(label, values):
    a = np.array(values)
    print(f"  {label:30s}  mean={a.mean():.1f}  median={np.median(a):.0f}  "
          f"min={a.min()}  max={a.max()}")

print(f"{len(scored_sessions)} sessions\n")
describe("Button presses per session", n_presses)
describe("Valid (has clip label)", n_valid)
describe("Invalid presses", n_invalid)
describe("Clip unknown", n_unknown)
describe("Unique clips recalled", n_unique)
describe("Total clip mentions (raw)", n_raw)

240 sessions

  Button presses per session      mean=17.6  median=16  min=0  max=75
  Valid (has clip label)          mean=16.5  median=15  min=0  max=74
  Invalid presses                 mean=0.7  median=0  min=0  max=22
  Clip unknown                    mean=0.4  median=0  min=0  max=9
  Unique clips recalled           mean=8.4  median=9  min=0  max=11
  Total clip mentions (raw)       mean=16.7  median=15  min=0  max=74


### Checking a multi-clip example

Some button presses are tagged with more than one clip (the participant recalled content from multiple clips in a single response). Let's see how common this is and what they look like:

In [13]:
multi_clip_count = 0
multi_clip_examples = []

for s in scored_sessions.values():
    for bp in s.presses:
        if len(bp.clip_numbers) > 1:
            multi_clip_count += 1
            if len(multi_clip_examples) < 5:
                multi_clip_examples.append((s.subject_code, bp))

total_presses = sum(len(s.presses) for s in scored_sessions.values())
print(f"{multi_clip_count} multi-clip presses out of {total_presses} total "
      f"({100 * multi_clip_count / total_presses:.1f}%)\n")

for code, bp in multi_clip_examples:
    print(f"  {code}: clips {bp.clip_numbers}")
    print(f"    {bp.text[:120]}\n")

29 multi-clip presses out of 4215 total (0.7%)

  BQFXW: clips [3, 8]
    drunk driving advert moments before man gets crushed by car

  CHHIO: clips [1, 2]
    Once again the woman teaching . girl throwing the beanbag, rethrowingit. I've also just remember that the guy in the caf

  FIPFF: clips [5, 2, 1]
    the man driving has a british accent and the teacher had an american accent , I noticed a number of accents in the coffe

  IDJRT: clips [8, 5]
    The other scene is about a man show the steps how to ride a forklift , he show the how to run it and make a turn . Anoth

  IDJRT: clips [11, 7]
    There are two scenes about the sport. The first one is two woman standing in the first then a womanshow the tutorial of 



The scored documents are now parsed into structured `ScoredSession` objects. Next stages (in future sections of this notebook) will:

- Parse the raw Excel exports for event-stream and cue data
- Match 5-letter subject codes to numeric participant IDs
- Assemble the full HDF5 dataset